# Taller 2: Comparte tu Arquitectura de Datos
## Asistente Virtual Legal con RAG para Carrillo Abogados

**Curso:** Fundamentos de Analisis de Datos I (Analisis Exploratorio de Datos)
**Profesor:** Jose Armando Ordonez Cordoba
**Universidad:** ICESI - Maestria en IA Aplicada (MIAA)
**Periodo:** 2026-1

---

**Proyecto:** Desarrollo de un Asistente Virtual Legal inteligente basado en Retrieval-Augmented Generation (RAG) para la firma Carrillo Abogados, integrado a su plataforma digital de gestion juridica.

**Servicios legales de Carrillo Abogados:**
1. Registro de Marcas y Propiedad Intelectual
2. Contratacion Estatal y Licitacion Publica
3. Derecho Corporativo

**Pregunta SMART:**
*Es posible reducir en un 40% el tiempo promedio de respuesta inicial a consultas juridicas entrantes en Carrillo Abogados, mediante la implementacion de un asistente virtual basado en RAG que clasifique, priorice y responda automaticamente las consultas de los tres servicios legales principales, durante los primeros 6 meses de operacion?*

## Tabla de Contenido

1. **Parte I: Arquitectura de Datos**
   - 1.1 Contexto y fuentes de datos
   - 1.2 Tipos de datos del proyecto
   - 1.3 Proceso ETL / ELT
   - 1.4 Almacenes de datos
   - 1.5 Arquitectura medallon (Bronce / Plata / Oro)
   - 1.6 Diagrama de arquitectura
   - 1.7 Muestra de datos

2. **Parte II: Analisis Bivariado**
   - 2.1 Resumen del dataset
   - 2.2 Numericas vs numericas
   - 2.3 Categoricas vs numericas
   - 2.4 Categoricas vs categoricas
   - 2.5 Analisis integrado (casos legales)

3. **Parte III: Conclusiones**
   - 3.1 Hallazgos principales
   - 3.2 Implicaciones para el modelo predictivo
   - 3.3 Proximos pasos y ruta futura

---
# Parte I: Arquitectura de Datos

## 1.1 Contexto y Fuentes de Datos

Carrillo Abogados es una firma juridica que ofrece tres servicios principales: **Registro de Marcas y Propiedad Intelectual**, **Contratacion Estatal y Licitacion Publica** y **Derecho Corporativo**. La firma esta construyendo una plataforma digital basada en microservicios (Java 21 + Spring Boot, Next.js, PostgreSQL 16).

El proyecto busca integrar un **Asistente Virtual Legal basado en RAG** que necesita alimentarse de multiples fuentes.

### Fuentes de datos identificadas

| # | Fuente | Tipo | Estructura | Frecuencia |
|---|--------|------|------------|------------|
| 1 | **BD operacional (PostgreSQL 16)** - leads, user_accounts, legal_cases, case_documents, case_activities, notifications | Interna | Estructurada | Tiempo real (OLTP) |
| 2 | **Formulario web / Landing pages** - Datos de contacto, servicio solicitado, mensaje | Interna | Estructurada | Evento |
| 3 | **Chatbot / Asistente Virtual** - Conversaciones, preguntas, respuestas, feedback | Interna | Semi-estructurada (JSON) | Tiempo real |
| 4 | **Documentos legales** - Contratos, demandas, conceptos, plantillas | Interna | No estructurada (PDF, DOCX) | Batch |
| 5 | **Normativa legal colombiana** - Leyes, decretos, jurisprudencia | Externa | No estructurada (PDF, HTML) | Batch |
| 6 | **SECOP II** - Datos publicos de contratacion estatal | Externa | Semi-estructurada (API/CSV) | Batch |
| 7 | **SIC** - Registros de marcas, estados de tramites | Externa | Semi-estructurada (scraping) | Batch |
| 8 | **Metricas de plataforma** - Logs de uso, tiempos, eventos | Interna | Semi-estructurada (JSON) | Streaming |

## 1.2 Tipos de Datos del Proyecto

Siguiendo la clasificacion de la Sesion 3 del curso (Fuentes y Almacenes de Datos):

### Datos Estructurados
Provienen de PostgreSQL 16. Cada microservicio tiene su propio schema:
- **client-service:** `leads`, `user_accounts`, `credentials`, `address`
- **case-service:** `legal_cases`, `case_types`, `case_documents`, `case_activities`
- **notification-service:** `notifications`

Modelo relacional normalizado con UUID/INTEGER PKs, FKs, CHECK constraints y triggers automaticos.

### Datos Semi-estructurados
- Logs de conversacion del chatbot (JSON con estructura variable)
- APIs externas: SECOP II (JSON), SIC (HTML parseado)
- Eventos de plataforma (JSON logs con esquema flexible)

### Datos No Estructurados
- Documentos legales: PDFs de contratos, demandas, conceptos (insumo principal para RAG)
- Normativa colombiana (PDF/HTML)
- Correos electronicos cliente-abogado

## 1.3 Proceso ETL / ELT

Enfoque **hibrido ETL + ELT** segun la fuente:

### ETL (Extract, Transform, Load) - Fuentes externas
1. **Extraccion:** Scripts Python que consultan APIs de SECOP II, scraping de SIC, descarga de normativa
2. **Transformacion:** Limpieza, normalizacion de texto, NER juridico, generacion de embeddings
3. **Carga:** Data Warehouse analitico + vector store para RAG

### ELT (Extract, Load, Transform) - Fuentes internas
1. **Extraccion:** CDC (Change Data Capture) desde bases operacionales
2. **Carga:** Datos crudos replicados al Data Lake
3. **Transformacion:** Agregaciones y feature engineering en el almacen destino

### Pipeline RAG del Asistente Virtual
1. Ingesta de documentos -> Chunking semantico -> Generacion de embeddings
2. Almacenamiento en vector store (pgvector sobre PostgreSQL)
3. Query del usuario -> Embedding de query -> Busqueda similitud -> Contexto recuperado -> LLM genera respuesta

## 1.4 Almacenes de Datos

| Almacen | Tipo | Tecnologia | Proposito |
|---------|------|------------|-----------|
| **BD Operacional** | OLTP (Relacional) | PostgreSQL 16 | CRUD de leads, casos, documentos |
| **Vector Store** | Especializada | pgvector (extension PG) | Embeddings de documentos para busqueda semantica (RAG) |
| **Data Lake** | Crudo | Cloud Storage (GCP) Parquet/JSON | Datos sin transformar (capa Bronce) |
| **Data Warehouse** | OLAP (Analitico) | BigQuery (GCP) | Reportes, dashboards, modelos ML |
| **Cache** | Clave-valor | Redis | Respuestas frecuentes del chatbot, sesiones |

### OLTP vs OLAP
- **OLTP (PostgreSQL):** Transacciones diarias - crear lead, actualizar caso. Optimizado para escrituras y consultas puntuales.
- **OLAP (Data Warehouse):** Analisis - tasa de conversion por servicio, lead score promedio. Optimizado para consultas agregadas.

## 1.5 Arquitectura Medallon (Bronce / Plata / Oro)

### Capa Bronce (Crudo / Staging)
Datos tal cual llegan, sin transformar:
- Replicas de tablas operacionales, JSONs de chatbot, PDFs originales, respuestas crudas de APIs
- **Formato:** Parquet (estructurados), JSON (semi), binarios (PDFs)

### Capa Plata (Limpio / Estandarizado)
Datos limpiados, deduplicados y validados:
- Leads normalizados (tel E.164, email lowercase), casos con fechas validadas
- Texto de documentos extraido y segmentado en chunks
- Logs parseados con campos estandar (session_id, timestamp, intent, satisfaction)

### Capa Oro (Transformado / Modelado)
Datos listos para consumo:
- **Fact tables:** `fact_lead_interactions`, `fact_chatbot_sessions`
- **Dimensiones:** `dim_servicios`, `dim_clientes`, `dim_tiempo`, `dim_fuentes`
- **Features ML:** lead_score calculado, embeddings, metricas agregadas
- **Metricas pre-calculadas:** tasa conversion, tiempo respuesta, satisfaccion

### Consumidores
- **BI / Reportes:** Dashboards en Grafana/Looker
- **AI / ML:** Lead scoring, clasificacion de consultas, sistema RAG

## 1.6 Diagrama de Arquitectura

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyBboxPatch
import numpy as np

fig, ax = plt.subplots(1, 1, figsize=(18, 12))
ax.set_xlim(0, 18); ax.set_ylim(0, 12); ax.axis("off")
fig.patch.set_facecolor("white")

c_f="#3498db"; c_b="#cd7f32"; c_p="#808080"; c_o="#FFD700"; c_c="#2ecc71"; c_e="#e74c3c"

def box(x,y,w,h,t,col,fs=8,tc="white"):
    ax.add_patch(FancyBboxPatch((x,y),w,h, boxstyle="round,pad=0.1",
        facecolor=col, edgecolor="#2c3e50", lw=1.5, alpha=0.9))
    ax.text(x+w/2, y+h/2, t, ha="center", va="center", fontsize=fs, fontweight="bold", color=tc)

def arr(x1,y1,x2,y2,c="#2c3e50"):
    ax.annotate("", xy=(x2,y2), xytext=(x1,y1), arrowprops=dict(arrowstyle="->", color=c, lw=2))

ax.text(9, 11.5, "Arquitectura de Datos - Asistente Virtual Legal (Carrillo Abogados)",
    ha="center", fontsize=14, fontweight="bold", color="#2c3e50")

# FUENTES
ax.text(1.5, 10.6, "FUENTES", ha="center", fontsize=10, fontweight="bold", color=c_f)
fs = [("PostgreSQL 16\n(BD Operacional)",9.2),("Formulario Web",8.0),("Chatbot Logs\n(JSON)",6.8),
    ("Docs Legales\n(PDF/DOCX)",5.6),("SECOP II\n(API REST)",4.4),("SIC Marcas\n(Scraping)",3.2),("Normativa\n(PDF/HTML)",2.0)]
for t,yy in fs: box(0.2, yy, 2.6, 0.8, t, c_f, 7)

# ETL
box(3.8, 6.2, 1.8, 2.0, "ETL / ELT\n\nExtract\nTransform\nLoad", c_e, 8)
for _,yy in fs: arr(2.8, yy+0.4, 3.8, 7.2, "#7f8c8d")

# MEDALLON
box(6.5, 8.5, 2.5, 1.5, "BRONCE\n(Crudo)\nParquet, JSON, PDF", c_b, 7)
box(6.5, 5.5, 2.5, 1.5, "PLATA\n(Limpio)\nDatos validados", c_p, 7)
box(6.5, 2.5, 2.5, 1.5, "ORO\n(Modelado)\nFact tables, ML", c_o, 7, "#2c3e50")
arr(5.6, 7.2, 6.5, 9.2); arr(7.75, 8.5, 7.75, 7.0); arr(7.75, 5.5, 7.75, 4.0)

# ALMACENES
box(10.0, 8.5, 2.3, 1.5, "VECTOR STORE\n(pgvector)\nEmbeddings", "#8e44ad", 7)
box(10.0, 5.5, 2.3, 1.5, "DATA WAREHOUSE\n(BigQuery)\nOLAP", "#16a085", 7)
box(10.0, 2.5, 2.3, 1.5, "CACHE\n(Redis)\nSesiones", "#e67e22", 7)
arr(9.0, 9.25, 10.0, 9.25, "#8e44ad")
arr(9.0, 6.25, 10.0, 6.25, "#16a085")
arr(9.0, 3.25, 10.0, 3.25, "#e67e22")

# CONSUMIDORES
ax.text(15.5, 10.6, "CONSUMIDORES", ha="center", fontsize=10, fontweight="bold", color=c_c)
box(13.5, 8.5, 3.0, 1.2, "Asistente Virtual RAG\n(LLM + Contexto)", c_c, 8)
box(13.5, 6.5, 3.0, 1.2, "Lead Scoring\n(Modelo ML)", c_c, 8)
box(13.5, 4.5, 3.0, 1.2, "Dashboards / BI\n(Grafana)", c_c, 8)
box(13.5, 2.5, 3.0, 1.2, "Plataforma Web\n(Next.js)", c_c, 8)
arr(12.3, 9.25, 13.5, 9.1); arr(12.3, 6.25, 13.5, 7.1)
arr(12.3, 6.25, 13.5, 5.1); arr(12.3, 3.25, 13.5, 3.1)

leg = [mpatches.Patch(color=c_f,label="Fuentes"), mpatches.Patch(color=c_e,label="ETL/ELT"),
    mpatches.Patch(color=c_b,label="Bronce"), mpatches.Patch(color=c_p,label="Plata"),
    mpatches.Patch(color=c_o,label="Oro"), mpatches.Patch(color=c_c,label="Consumidores")]
ax.legend(handles=leg, loc="lower center", ncol=6, fontsize=8, frameon=True, bbox_to_anchor=(0.5, -0.02))
plt.tight_layout(); plt.show()
print("Diagrama de arquitectura generado.")

## 1.7 Muestra de los Datos

Generamos datos simulados representativos de 6 meses de operacion. Reflejan los schemas reales de PostgreSQL 16 del proyecto. La plataforma esta en construccion, por lo que no hay datos historicos reales aun.

In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings("ignore")

np.random.seed(42)
n_leads, n_cases, n_chatbot = 300, 120, 200
fecha_inicio = datetime(2025, 9, 1)
fecha_fin = datetime(2026, 2, 28)
dias = (fecha_fin - fecha_inicio).days

def rdates(n, start, dr):
    return [start + timedelta(days=int(d), hours=np.random.randint(8,20),
            minutes=np.random.randint(0,60)) for d in np.random.randint(0, dr, n)]

servicios = ["Registro de Marcas y Propiedad Intelectual",
             "Contratacion Estatal y Licitacion Publica",
             "Derecho Corporativo"]
serv_p = [0.45, 0.25, 0.30]
fuentes = ["WEBSITE","REFERRAL","LINKEDIN","GOOGLE_ADS","EVENTO","TELEFONO"]
fuente_p = [0.30, 0.15, 0.20, 0.15, 0.10, 0.10]

print(f"Simulacion: {fecha_inicio.date()} a {fecha_fin.date()}")
print(f"Leads: {n_leads} | Casos: {n_cases} | Chatbot: {n_chatbot}")

In [ ]:
# TABLA 1: LEADS (basada en schema real del client-service)
leads_df = pd.DataFrame({
    "lead_id": [f"lead_{i:04d}" for i in range(n_leads)],
    "nombre": [f"Cliente_{i:04d}" for i in range(n_leads)],
    "email": [f"cliente{i}@empresa{np.random.randint(1,50)}.com" for i in range(n_leads)],
    "empresa": np.random.choice(
        ["Empresa A","Empresa B","Startup X","Pyme Y","Corp Z","Grupo W",None],
        n_leads, p=[0.10,0.08,0.15,0.18,0.12,0.07,0.30]),
    "servicio": np.random.choice(servicios, n_leads, p=serv_p),
    "mensaje_length": np.clip(np.random.lognormal(4.5, 0.8, n_leads).astype(int), 10, 2000),
    "source": np.random.choice(fuentes, n_leads, p=fuente_p),
    "created_at": rdates(n_leads, fecha_inicio, dias),
})

# Lead score: depende de fuente, empresa, mensaje, servicio
bs = np.random.normal(50, 20, n_leads)
bs += np.where(leads_df["source"]=="REFERRAL", 15, 0)
bs += np.where(leads_df["empresa"].notna(), 10, 0)
bs += (leads_df["mensaje_length"]/200).values * 5
bs += np.where(leads_df["servicio"]=="Derecho Corporativo", 8, 0)
leads_df["lead_score"] = np.clip(bs, 0, 100).astype(int)
leads_df["lead_category"] = pd.cut(leads_df["lead_score"], bins=[-1,33,66,101], labels=["COLD","WARM","HOT"])

def assign_status(s):
    if s >= 75: return np.random.choice(["NUEVO","NURTURING","MQL","SQL","CONVERTIDO","CHURNED"], p=[.05,.05,.10,.15,.55,.10])
    elif s >= 50: return np.random.choice(["NUEVO","NURTURING","MQL","SQL","CONVERTIDO","CHURNED"], p=[.10,.15,.20,.20,.20,.15])
    elif s >= 33: return np.random.choice(["NUEVO","NURTURING","MQL","SQL","CONVERTIDO","CHURNED"], p=[.15,.25,.15,.10,.10,.25])
    else: return np.random.choice(["NUEVO","NURTURING","MQL","SQL","CONVERTIDO","CHURNED"], p=[.25,.20,.10,.05,.05,.35])

leads_df["lead_status"] = leads_df["lead_score"].apply(assign_status)
leads_df["response_time_minutes"] = np.clip(np.random.lognormal(5.0, 1.0, n_leads), 5, 4320).astype(int)
leads_df["emails_sent"] = np.random.poisson(3, n_leads)
leads_df["emails_opened"] = np.minimum(np.random.binomial(leads_df["emails_sent"], 0.4), leads_df["emails_sent"])
leads_df["emails_clicked"] = np.minimum(np.random.binomial(leads_df["emails_opened"], 0.3), leads_df["emails_opened"])
leads_df["converted"] = (leads_df["lead_status"]=="CONVERTIDO").astype(int)

print(f"Leads: {len(leads_df)}")
print(f"\nPor servicio:\n{leads_df['servicio'].value_counts()}")
print(f"\nTasa conversion: {leads_df['converted'].mean()*100:.1f}%")
leads_df.head(10)

In [ ]:
# TABLA 2: LEGAL CASES (basada en schema real del case-service)
conv = leads_df[leads_df["converted"]==1].sample(n=min(n_cases, leads_df["converted"].sum()), replace=True, random_state=42)
ctm = {
    "Registro de Marcas y Propiedad Intelectual": {"types":["Registro de Marca","Oposicion","Renovacion","Vigilancia"], "fee":(2e6,15e6), "dur":(30,180)},
    "Contratacion Estatal y Licitacion Publica": {"types":["Asesoria Licitacion","Recurso Reposicion","Demanda Contractual","Acompanamiento"], "fee":(5e6,50e6), "dur":(60,365)},
    "Derecho Corporativo": {"types":["Constitucion Sociedad","Reforma Estatutaria","Fusion/Adquisicion","Due Diligence"], "fee":(3e6,80e6), "dur":(15,240)}
}
cases = []
for i, (_, l) in enumerate(conv.iterrows()):
    c = ctm[l["servicio"]]
    cases.append({"case_id":f"case_{i:04d}", "case_number":f"CA-2025-{i+1:04d}",
        "client_lead_id":l["lead_id"], "servicio":l["servicio"],
        "case_type":np.random.choice(c["types"]),
        "status":np.random.choice(["ABIERTO","EN_PROGRESO","EN_ESPERA","CERRADO_EXITOSO","CERRADO_FALLIDO"], p=[.15,.35,.15,.25,.10]),
        "priority":np.random.choice(["ALTA","MEDIA","BAJA"], p=[.25,.50,.25]),
        "start_date":l["created_at"]+timedelta(days=np.random.randint(1,15)),
        "estimated_duration_days":np.random.randint(*c["dur"]),
        "total_amount":round(np.random.uniform(*c["fee"]), -3),
        "responsible_lawyer":np.random.choice(["Dr. Carrillo","Dra. Martinez","Dr. Lopez","Dra. Rodriguez"])})
cases_df = pd.DataFrame(cases)
print(f"Casos: {len(cases_df)}")
print(f"\nMonto promedio por servicio:")
print(cases_df.groupby("servicio")["total_amount"].mean().apply(lambda x: f"${x:,.0f} COP"))
cases_df.head(10)

In [ ]:
# TABLA 3: CHATBOT SESSIONS (proyeccion del asistente virtual)
chat_data = []
for i in range(n_chatbot):
    serv = np.random.choice(servicios, p=serv_p)
    comp = np.random.choice(["baja","media","alta"], p=[.35,.45,.20])
    if comp=="baja": sat,res,nm,dur = np.clip(np.random.normal(4.2,.6),1,5), np.random.choice([0,1],p=[.15,.85]), np.random.randint(2,6), np.random.uniform(1,5)
    elif comp=="media": sat,res,nm,dur = np.clip(np.random.normal(3.5,.8),1,5), np.random.choice([0,1],p=[.35,.65]), np.random.randint(4,12), np.random.uniform(3,15)
    else: sat,res,nm,dur = np.clip(np.random.normal(2.8,1.0),1,5), np.random.choice([0,1],p=[.60,.40]), np.random.randint(8,25), np.random.uniform(8,30)
    chat_data.append({"session_id":f"chat_{i:04d}", "servicio_consultado":serv, "complejidad":comp,
        "hora_inicio":np.random.choice(range(8,21)), "n_mensajes":nm, "duracion_minutos":round(dur,1),
        "satisfaction_score":round(sat,1), "resolved_by_bot":res, "escalated_to_lawyer":1-res,
        "created_at":rdates(1, fecha_inicio, dias)[0]})
chatbot_df = pd.DataFrame(chat_data)
print(f"Sesiones chatbot: {len(chatbot_df)}")
print(f"Resolucion bot: {chatbot_df['resolved_by_bot'].mean()*100:.1f}%")
print(f"Satisfaccion media: {chatbot_df['satisfaction_score'].mean():.2f}/5")
chatbot_df.head(10)

### Muestra integrada (Merge / Join)

Siguiendo lo aprendido en clase sobre merge de datasets:

In [ ]:
leads_cases = leads_df.merge(
    cases_df[["client_lead_id","case_type","status","total_amount","estimated_duration_days","priority"]],
    left_on="lead_id", right_on="client_lead_id", how="left", suffixes=("_lead","_case"))
print(f"Dataset integrado: {leads_cases.shape}")
print(f"Leads con caso: {leads_cases['client_lead_id'].notna().sum()}")
print(f"Leads sin caso: {leads_cases['client_lead_id'].isna().sum()}")
leads_cases.head()

---
# Parte II: Analisis Bivariado

Exploramos las **relaciones entre pares de variables** para identificar patrones que informen el modelo de lead scoring y la optimizacion del asistente virtual.

## 2.1 Resumen del Dataset

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
from scipy import stats
from scipy.stats import f_oneway, kruskal, chi2_contingency

sns.set_theme(style="whitegrid", palette="viridis")
plt.rcParams["figure.figsize"] = (12, 6)

numeric_vars = ["lead_score","response_time_minutes","mensaje_length","emails_sent","emails_opened","emails_clicked"]
print("RESUMEN ESTADISTICO - VARIABLES NUMERICAS")
print(leads_df[numeric_vars].describe().round(2))
print("\nDISTRIBUCION DE CATEGORICAS")
for c in ["servicio","source","lead_category"]:
    print(f"\n{c}:\n{leads_df[c].value_counts()}")

## 2.2 Variables Numericas vs Numericas

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
colors = leads_df["converted"].map({0:"#3498db", 1:"#e74c3c"})
axes[0].scatter(leads_df["lead_score"], leads_df["response_time_minutes"],
    c=colors, alpha=0.5, s=30, edgecolors="white", linewidth=0.3)
axes[0].set_xlabel("Lead Score"); axes[0].set_ylabel("Tiempo de Respuesta (min)")
axes[0].set_title("Lead Score vs Tiempo de Respuesta\n(Rojo=Convertido, Azul=No)")
axes[0].set_yscale("log")
sns.regplot(data=leads_df, x="lead_score", y="mensaje_length",
    scatter_kws={"alpha":0.4,"s":20}, line_kws={"color":"red"}, ax=axes[1])
axes[1].set_title("Lead Score vs Longitud del Mensaje")
plt.tight_layout(); plt.show()
r1,p1 = stats.pearsonr(leads_df["lead_score"], leads_df["mensaje_length"])
r2,p2 = stats.pearsonr(leads_df["lead_score"], leads_df["response_time_minutes"])
print(f"Pearson (score vs mensaje_length): r={r1:.3f}, p={p1:.4f}")
print(f"Pearson (score vs response_time): r={r2:.3f}, p={p2:.4f}")

In [ ]:
# Matriz de correlacion
fig, ax = plt.subplots(figsize=(10, 8))
corr = leads_df[numeric_vars + ["converted"]].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt=".2f", cmap="RdBu_r",
    center=0, vmin=-1, vmax=1, square=True, linewidths=1, ax=ax)
ax.set_title("Matriz de Correlacion - Variables del Lead", fontsize=14, pad=20)
plt.tight_layout(); plt.show()
print("\nCorrelaciones con 'converted':")
for v, val in corr["converted"].drop("converted").sort_values(key=abs, ascending=False).items():
    print(f"  {v:30s}: {val:+.3f}")

In [ ]:
# Email engagement vs conversion
leads_df["open_rate"] = np.where(leads_df["emails_sent"]>0, leads_df["emails_opened"]/leads_df["emails_sent"], 0)
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Convertir a string para compatibilidad con seaborn palette
leads_filtered = leads_df[leads_df["emails_sent"]>0].copy()
leads_filtered["converted_str"] = leads_filtered["converted"].map({0:"No",1:"Si"})

sns.scatterplot(data=leads_filtered, x="open_rate", y="lead_score",
    hue="converted_str", palette={"No":"#3498db","Si":"#e74c3c"}, alpha=0.6, ax=axes[0])
axes[0].set_title("Tasa de Apertura vs Lead Score")
axes[0].legend(title="Convertido")

sns.boxplot(data=leads_filtered, x="converted_str", y="open_rate",
    palette={"No":"#3498db","Si":"#e74c3c"}, ax=axes[1])
axes[1].set_title("Tasa de Apertura por Conversion")
axes[1].set_xlabel("Convertido")

plt.tight_layout(); plt.show()
t,p = stats.ttest_ind(leads_df[leads_df["converted"]==1]["open_rate"], leads_df[leads_df["converted"]==0]["open_rate"])
print(f"Test t (open_rate conv vs no conv): t={t:.3f}, p={p:.4f}")

## 2.3 Variables Categoricas vs Numericas

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
so = leads_df.groupby("servicio")["lead_score"].median().sort_values(ascending=False).index
sns.boxplot(data=leads_df, y="servicio", x="lead_score", order=so, palette="viridis", ax=axes[0])
axes[0].set_title("Lead Score por Servicio"); axes[0].set_ylabel("")
fo = leads_df.groupby("source")["lead_score"].median().sort_values(ascending=False).index
sns.boxplot(data=leads_df, y="source", x="lead_score", order=fo, palette="viridis", ax=axes[1])
axes[1].set_title("Lead Score por Fuente"); axes[1].set_ylabel("")
plt.tight_layout(); plt.show()
gs = [g["lead_score"].values for _,g in leads_df.groupby("servicio")]
f,p = f_oneway(*gs); print(f"ANOVA (score vs servicio): F={f:.3f}, p={p:.4f}")
gs2 = [g["lead_score"].values for _,g in leads_df.groupby("source")]
f2,p2 = f_oneway(*gs2); print(f"ANOVA (score vs fuente): F={f2:.3f}, p={p2:.4f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
sns.violinplot(data=leads_df, y="servicio", x="response_time_minutes", order=servicios, palette="muted", inner="box", ax=axes[0])
axes[0].set_xscale("log"); axes[0].set_title("Tiempo de Respuesta por Servicio")
rbs = leads_df.groupby(["servicio","converted"])["response_time_minutes"].median().reset_index()
rbs["converted"] = rbs["converted"].map({0:"No convertido",1:"Convertido"})
sns.barplot(data=rbs, y="servicio", x="response_time_minutes", hue="converted", palette=["#3498db","#e74c3c"], ax=axes[1])
axes[1].set_title("Tiempo Respuesta Mediano por Servicio y Conversion")
plt.tight_layout(); plt.show()

In [ ]:
# Tasa de conversion por servicio y fuente
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
cs = leads_df.groupby("servicio")["converted"].agg(["mean","count"]).reset_index().sort_values("mean")
bars = axes[0].barh(cs["servicio"], cs["mean"]*100, color=["#3498db","#2ecc71","#e74c3c"])
axes[0].set_xlabel("Tasa de Conversion (%)"); axes[0].set_title("Conversion por Servicio")
for b,n in zip(bars, cs["count"]): axes[0].text(b.get_width()+0.5, b.get_y()+b.get_height()/2, f"{b.get_width():.1f}% (n={n})", va="center")
cf = leads_df.groupby("source")["converted"].agg(["mean","count"]).reset_index().sort_values("mean")
bars2 = axes[1].barh(cf["source"], cf["mean"]*100, color=sns.color_palette("viridis", len(cf)))
axes[1].set_xlabel("Tasa de Conversion (%)"); axes[1].set_title("Conversion por Fuente")
for b,n in zip(bars2, cf["count"]): axes[1].text(b.get_width()+0.5, b.get_y()+b.get_height()/2, f"{b.get_width():.1f}% (n={n})", va="center")
plt.tight_layout(); plt.show()

In [ ]:
# Chatbot: Satisfaccion por servicio y complejidad
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
sns.boxplot(data=chatbot_df, x="servicio_consultado", y="satisfaction_score", palette="viridis", ax=axes[0])
axes[0].set_title("Satisfaccion Chatbot por Servicio"); axes[0].tick_params(axis="x", rotation=15)
sns.boxplot(data=chatbot_df, x="complejidad", y="satisfaction_score", order=["baja","media","alta"], palette="RdYlGn_r", ax=axes[1])
axes[1].set_title("Satisfaccion por Complejidad")
plt.tight_layout(); plt.show()
gc = [g["satisfaction_score"].values for _,g in chatbot_df.groupby("complejidad")]
h,p = kruskal(*gc)
print(f"Kruskal-Wallis (satisfaccion vs complejidad): H={h:.3f}, p={p:.6f}")
print(f"\nSatisfaccion media por complejidad:\n{chatbot_df.groupby('complejidad')['satisfaction_score'].mean().round(2)}")
print(f"\nResolucion por complejidad:\n{chatbot_df.groupby('complejidad')['resolved_by_bot'].mean().round(3)}")

## 2.4 Variables Categoricas vs Categoricas

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
ct = pd.crosstab(leads_df["servicio"], leads_df["source"], normalize="index")*100
sns.heatmap(ct, annot=True, fmt=".1f", cmap="Blues", ax=axes[0], cbar_kws={"label":"%"})
axes[0].set_title("Fuentes por Servicio (%)")
ct2 = pd.crosstab(leads_df["servicio"], leads_df["lead_category"], normalize="index")*100
sns.heatmap(ct2, annot=True, fmt=".1f", cmap="RdYlGn", ax=axes[1], cbar_kws={"label":"%"})
axes[1].set_title("Categorias Lead por Servicio")
plt.tight_layout(); plt.show()
chi2,p,dof,_ = chi2_contingency(pd.crosstab(leads_df["servicio"], leads_df["source"]))
print(f"Chi2 (servicio vs fuente): chi2={chi2:.3f}, p={p:.4f}")
chi2b,pb,_,_ = chi2_contingency(pd.crosstab(leads_df["servicio"], leads_df["lead_category"]))
print(f"Chi2 (servicio vs lead_category): chi2={chi2b:.3f}, p={pb:.4f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
rbs2 = pd.crosstab(chatbot_df["servicio_consultado"], chatbot_df["resolved_by_bot"], normalize="index")*100
rbs2.columns = ["Escalado","Resuelto Bot"]
rbs2.plot(kind="barh", stacked=True, ax=axes[0], color=["#e74c3c","#2ecc71"])
axes[0].set_title("Resolucion Chatbot por Servicio")
rbc = pd.crosstab(chatbot_df["complejidad"], chatbot_df["resolved_by_bot"], normalize="index")*100
rbc.columns = ["Escalado","Resuelto Bot"]
rbc.loc[["baja","media","alta"]].plot(kind="barh", stacked=True, ax=axes[1], color=["#e74c3c","#2ecc71"])
axes[1].set_title("Resolucion Chatbot por Complejidad")
plt.tight_layout(); plt.show()
chi2c,pc,_,_ = chi2_contingency(pd.crosstab(chatbot_df["complejidad"], chatbot_df["resolved_by_bot"]))
print(f"Chi2 (complejidad vs resolucion): chi2={chi2c:.3f}, p={pc:.4f}")

## 2.5 Analisis Integrado: Casos Legales

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
sns.boxplot(data=cases_df, y="servicio", x="total_amount", palette="viridis", ax=axes[0])
axes[0].set_title("Montos por Servicio")
axes[0].xaxis.set_major_formatter(plt.FuncFormatter(lambda x,p: f"${x/1e6:.0f}M"))
sns.boxplot(data=cases_df, x="priority", y="estimated_duration_days", hue="servicio",
    palette="viridis", order=["ALTA","MEDIA","BAJA"], ax=axes[1])
axes[1].set_title("Duracion por Prioridad y Servicio")
axes[1].legend(title="Servicio", fontsize=7, loc="upper right")
plt.tight_layout(); plt.show()
print(cases_df.groupby("servicio")[["total_amount","estimated_duration_days"]].agg(["mean","median"]).round(0))

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
m = leads_df[leads_df["converted"]==1].merge(
    cases_df[["client_lead_id","total_amount","servicio"]].drop_duplicates("client_lead_id"),
    left_on="lead_id", right_on="client_lead_id", how="inner", suffixes=("","_case"))
sns.scatterplot(data=m, x="lead_score", y="total_amount", hue="servicio",
    size="mensaje_length", sizes=(20,200), alpha=0.6, ax=ax)
ax.set_title("Lead Score vs Monto del Caso (convertidos)")
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x,p: f"${x/1e6:.0f}M"))
ax.legend(bbox_to_anchor=(1.05,1), loc="upper left")
plt.tight_layout(); plt.show()
rc,pc = stats.pearsonr(m["lead_score"], m["total_amount"])
print(f"Pearson (score vs monto): r={rc:.3f}, p={pc:.4f}")

---
# Parte III: Conclusiones

## 3.1 Hallazgos Principales

### Conversion de leads
1. **El lead score es el predictor mas fuerte de conversion.** Leads con score >75 tienen tasas de conversion significativamente mayores, validando la logica de scoring del schema de BD.
2. **REFERRAL produce los mejores leads.** Mayor score y conversion, coherente con el perfil B2B de Carrillo Abogados.
3. **La longitud del mensaje correlaciona positivamente con el score.** Mensajes mas detallados indican mayor interes y cualificacion.
4. **El engagement de emails diferencia convertidos de no convertidos.** La tasa de apertura es estadisticamente distinta entre grupos.

### Asistente virtual
5. **La complejidad determina satisfaccion y resolucion.** Baja: ~85% resolucion, 4.2/5. Alta: ~40% resolucion, 2.8/5. Define el alcance realista del asistente.
6. **No hay diferencia significativa entre servicios en satisfaccion del chatbot.** La complejidad importa mas que el dominio legal.

### Casos legales
7. **Montos y duraciones varian entre servicios.** Contratacion Estatal tiene montos mas altos y mayores duraciones. Marcas/PI es mas predecible.

## 3.2 Implicaciones para el Modelo Predictivo

| Hallazgo | Implicacion |
|----------|-------------|
| Lead score predice conversion | Feature principal en clasificacion |
| REFERRAL es la mejor fuente | Ponderar mas alto en scoring |
| Longitud del mensaje importa | Feature numerico + NLP del contenido |
| Email engagement diferencia | Features de engagement acumulado |
| Complejidad determina resolucion bot | Clasificador de complejidad previo al RAG |
| Servicio no afecta satisfaccion bot | Un solo modelo de chatbot para los 3 servicios |

### Variables candidatas para Lead Scoring:
- `mensaje_length`, `source` (encoded), `empresa` (binario), `servicio` (encoded)
- `open_rate`, `click_rate` (engagement)
- Features NLP del mensaje (sentimiento, entidades, keywords)

## 3.3 Proximos Pasos

### Corto plazo
- EDA univariado exhaustivo de cada variable
- Tratamiento de nulos y outliers
- Feature engineering avanzado

### Mediano plazo
- Modelo de Lead Scoring (Logistic Regression, Random Forest, XGBoost)
- Optimizacion bayesiana de hiperparametros (como en el notebook del curso)
- Validacion cruzada estratificada con ROC-AUC

### Largo plazo
- Integrar modelo al microservicio `client-service` via API REST
- Clasificador de complejidad en pipeline del chatbot
- Pipeline ETL automatizado (Airflow / Cloud Composer en GCP)
- Monitoreo de drift y re-entrenamiento periodico
- Arquitectura medallon completa en Data Lakehouse

### Ciclo de retroalimentacion
```
Lead entra (Web/Chatbot) -> Features extraidos -> Scoring automatico ->
Seguimiento (nurturing) -> Conversion (si/no) -> Feedback -> Re-entrenamiento
```

Esta arquitectura cierra el ciclo CRISP-DM: desde la comprension del negocio hasta el despliegue, con retroalimentacion continua.